# 02 - Limpieza y Normalización de Datos

En este segundo reto, aplicaremos técnicas de limpieza para transformar el dataset de giras musicales en un formato estructurado y listo para el análisis. Nos enfocaremos en corregir formatos de moneda, manejar caracteres especiales y normalizar años.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Carga de datos
data_path = Path("..") / "data" / "my_file (1).csv"
df = pd.read_csv(data_path)

print(f"Dimensiones iniciales: {df.shape}")

## 1. Limpieza de Moneda y Caracteres Especiales

Corregiremos las columnas `Actual gross` y `Adjusted gross (in 2022 dollars)` eliminando símbolos como `$`, `,` y referencias entre corchetes (ej. `[b]`).

In [ ]:
def clean_currency(value):
    if pd.isna(value):
        return np.nan
    # Eliminar símbolos de moneda, comas y referencias entre corchetes
    cleaned = re.sub(r'[\$,\[\]a-zA-Z\s†‡*]', '', str(value))
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

currency_cols = ['Actual gross', 'Adjusted gross (in 2022 dollars)', 'Average gross']
for col in currency_cols:
    df[col] = df[col].apply(clean_currency)

df[currency_cols].head()

## 2. Normalización de Años y Títulos

Eliminaremos caracteres especiales en los títulos y normalizaremos el formato de los años.

In [ ]:
def clean_titles(text):
    if pd.isna(text):
        return text
    # Eliminar símbolos especiales comunes en el dataset
    return re.sub(r'[†‡*\s\[\]a-zA-Z\d]*\[\d+\]', '', str(text)).strip()

df['Tour title'] = df['Tour title'].apply(clean_titles)
df['Year(s)'] = df['Year(s)'].apply(lambda x: re.sub(r'\D+', '-', str(x)) if pd.notna(x) else x)

df[['Artist', 'Tour title', 'Year(s)']].head()

## 3. Tratamiento de Valores Nulos

Imputaremos valores razonables para los rankings y eliminaremos filas críticas vacías.

In [ ]:
# Reemplazar NaNs en rankings con el valor máximo para representar posiciones bajas
df['Peak'] = pd.to_numeric(df['Peak'].astype(str).str.extract('(\d+)', expand=False)).fillna(df.index.max() + 1)

# Guardar dataset limpio
output_dir = Path("..") / "data"
df.to_csv(output_dir / "music_tours_cleaned.csv", index=False)
print("Dataset guardado en: data/music_tours_cleaned.csv")